Concept 1 — Parameterized Notebooks

A notebook should not be hard-coded.Instead of always generating the Silver layer, a notebook can accept parameters 
like: layer = bronze
layer = silver

This makes the same notebook reusable in pipelines.

Databricks provides widgets for this.

In [0]:
# Step 1 — Create Pipeline Parameter

dbutils.widgets.text("layer", "silver")
layer = dbutils.widgets.get("layer").lower().strip()

print("Running pipeline layer:", layer)
# This creates aruntime parameter called layer. Default value: silver
# If the job later runs with: layer=bronze , the notebook will behave differently.This allows one notebook to control multiple pipeline stages.


Running pipeline layer: silver


In [0]:
# Step 2 — Import Section
from pyspark.sql import functions as F



## **Pipeline Functions(Core Logic)**

In [0]:
# Bronze Pipeline -- This function: • Ingests raw data • Writes Bronze Delta table
# ---------------------------------------
# Bronze Pipeline
# ---------------------------------------

def run_bronze():

    print("Running Bronze ingestion pipeline")

    events = spark.read.csv(
        "/Volumes/workspace/ecommerce/ecommerce_data/*.csv",
        header=True,
        inferSchema=True
    )

    events.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("workspace.ecommerce.bronze_events_raw")

In [0]:
# Silver Pipeline Function-- This function: • Reads Bronze • Creates feature engineering layer
# ---------------------------------------
# Silver Pipeline
# ---------------------------------------

def run_silver():

    print("Running Silver feature engineering pipeline")

    bronze_df = spark.table("workspace.ecommerce.bronze_events")

    user_features_df = bronze_df.groupBy("user_id").agg(

        F.count("*").alias("total_events"),
        F.count(F.when(F.col("event_type") == "purchase", True)).alias("total_purchases"),
        F.sum("price").alias("total_spent"),
        F.avg("price").alias("avg_price")

    )

    user_features_df.write \
        .format("delta") \
            .mode("overwrite") \
                .option("overwriteSchema","true") \
                    .saveAsTable("workspace.ecommerce.silver_user_features")

In [0]:
# ---------------------------------------
# Pipeline Controller
# ---------------------------------------

if layer == "bronze":

    run_bronze()

elif layer == "silver":

    run_silver()

else:

    print("Unknown pipeline layer")

Running Silver feature engineering pipeline
